In [ ]:
# for google colab
# Clone the repo and set paths
REPO_URL = "https://github.com/MadKeyboardArtist/5703-Federated-Model.git"
REPO_DIR = "/content/5703-Federated-Model"

import os, sys
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
os.chdir(REPO_DIR)
sys.path.append(REPO_DIR)   # so Python can import your modules
print("CWD:", os.getcwd())

'\n# for google colab\n# GPU check\nimport torch, subprocess, textwrap\nprint("CUDA available:", torch.cuda.is_available())\n!nvidia-smi\n\n# Clone the repo and set paths\nREPO_URL = "https://github.com/MadKeyboardArtist/5703-Federated-Model.git"\nREPO_DIR = "/content/5703-Federated-Model"\n\nimport os, sys\nif not os.path.exists(REPO_DIR):\n    !git clone {REPO_URL}\nos.chdir(REPO_DIR)\nsys.path.append(REPO_DIR)   # so Python can import your modules\nprint("CWD:", os.getcwd())\n'

In [ ]:
# for google colab
# GPU check
import torch, subprocess, textwrap
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi

In [2]:
# initialize model

# Per Round:
# server copies and sends the global model weights.
# client trains locally (on their local dataset).
# client returns its new weights and sample count.
# server aggregates the weights using FedAvg.
# global model is updated.
# (Optional) Evaluate global model on a held-out test set.


# heads record:
# 1. always save the newest
# 2. always save the best ever
# 3. always save curretn best

In [3]:
# external libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import json
import shutil
import pandas as pd

from collections import defaultdict

In [4]:
# self-built files
from federated_multihead_model import SharedEncoders, TabularClientModel, ImageClientModel, MultiClientModel
from config import D_TABULAR, D_EMBEDDING, D_FUSION

from tabular_site_training import training as complete_tabular_training

#from image_site_training   import training as complete_image_training
'''
from multi_site_training   import training_loop as multi_training_loop
'''

'\nfrom multi_site_training   import training_loop as multi_training_loop\n'

In [5]:
# reimport self-built files
import importlib
import federated_multihead_model
import tabular_site_training
importlib.reload(federated_multihead_model)
importlib.reload(tabular_site_training)

<module 'tabular_site_training' from 'd:\\USYD\\2025 S2\\5703 Capstone\\model\\tabular_site_training.py'>

In [6]:
# 1. build local model: global encoders + local heads:
def build_local_head (folder_name, model_name, modality, n_classes):
    # 1. define the complete model structure
    # solved with import
    
    # 2. assemble local model
    # 2.1 Initialize local encoders
    encoder_placeholder = SharedEncoders(
        d_tabular   = D_TABULAR, 
        d_embedding = D_EMBEDDING, 
        d_fusion    = D_FUSION
        )
    # 2.2 Load the weights
    # no need here

    # 2.3 build local model
    # modality check
    if modality == "tabular":
        local_model = TabularClientModel(shared_encoders = encoder_placeholder, n_classes = n_classes)
    elif modality == "image":
        local_model = ImageClientModel  (shared_encoders = encoder_placeholder, n_classes = n_classes)
    elif modality == "multi":
        local_model = MultiClientModel  (shared_encoders = encoder_placeholder, n_classes = n_classes)
    else:
        # report ERROR
        exit()
    
    # 2.4 save the local head weights
    os.makedirs(folder_name, exist_ok = True)
    head_path = os.path.join(folder_name, f"{model_name}.pth")
    torch.save(local_model.head.state_dict(), head_path)
    return head_path

In [7]:
def assign_local_training_function (modality):
    if modality == "tabular":
        return complete_tabular_training
    elif modality == "image":
        pass
        # return complete_image_training
    elif modality == "multi":
        pass
    else:
        # report ERROR
        return None

In [8]:
def global_agg_fedavg(client_global_updates):
    '''
    def selective_fedavg(client_states, client_weights):
        """FedAvg per key (only keys that were returned by clients)"""

        grouped = defaultdict(list)

        for state, weight in zip(client_states, client_weights):
            for k, v in state.items():
                grouped[k].append((v, weight))

        avg_state = {}
        for k, updates in grouped.items():
            total_weight = sum(w for _, w in updates)
            avg_state[k] = sum((v * (w / total_weight)) for v, w in updates)

        return avg_state
    '''
    return client_global_updates[0][0]

In [ ]:
'''
[{'name': 'tabular_1',
  'modality': 'tabular',
  'raw_dataset_path': 'tabular_dataset/diabetes_012_ready_to_model.csv',
  'clean_dataset': 'tabular_dataset/diabetes_012_ready_to_model.csv',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'tabular_dataset/diabetes_012_train.csv',
  'clean_dataset_val': 'tabular_dataset/diabetes_012_val.csv',
  'clean_dataset_test': 'tabular_dataset/diabetes_012_test.csv',
  'newest_head': 'newest_local_heads\\tabular_1.pth',
  'overall_best_head': 'overall_best_local_heads\\tabular_1.pth',
  'current_best_head': 'current_best_local_heads\\tabular_1.pth',
  'site_training': <function tabular_site_training.training(global_state, train_set_path, val_set_path, labelcol, n_classes, newest_head_path, current_best_head_path)>,
  'head_re_training': <function tabular_site_training.final_head_training(global_state, train_set_path, val_set_path, labelcol, n_classes, newest_head_path)>,
  'evaluation': <function tabular_site_training.evaluation(global_state, val_set_path, labelcol, n_classes, newest_head_path)>}]
'''
def local_training (global_state, site, freeze_global = False):
    if site["modality"] == "tabular":
        site_name              = site["name"]
        train_set_path         = site["clean_dataset_train"]
        val_set_path           = site["clean_dataset_val"]
        label_col_name         = site["label_col"]
        num_of_classes         = site["n_classes"]
        newest_head_path       = site["newest_head"] 
        current_best_head_path = site["current_best_head"]
        freeze_global          = freeze_global

        return site["site_training"](
            site_name              = site_name,
            global_state           = global_state,
            freeze_global          = freeze_global,
            train_set_path         = train_set_path,
            val_set_path           = val_set_path, 
            labelcol               = label_col_name, 
            n_classes              = num_of_classes, 
            newest_head_path       = newest_head_path,
            current_best_head_path = current_best_head_path
            )
    
    elif site["modality"] == "image":
        site_name              = site["name"]
        train_set_path         = site["clean_dataset_train"]
        val_set_path           = site["clean_dataset_val"]
        num_of_classes         = site["n_classes"]
        newest_head_path       = site["newest_head"] 
        current_best_head_path = site["current_best_head"]
        freeze_global          = freeze_global

        return site["site_training"](
            site_name              = site_name,
            global_state           = global_state,
            freeze_global          = freeze_global,
            train_set_path         = train_set_path,
            val_set_path           = val_set_path, 
            n_classes              = num_of_classes, 
            newest_head_path       = newest_head_path,
            current_best_head_path = current_best_head_path
            )

    elif site["modality"] == "multi":
        return None
    
    else:
        print("site modality error")
        exit()

In [11]:
def federated_training_one_round (global_state, sites, freeze_global):
    # 1. sever send global encoder weights to sites
    # by passing global_state
    
    # 2. full local training
    ################ COME TO EACH SITES ##################
    global_updates_with_sample_count = []
    all_ckpt_eva_results = []
    # current_best_client_heads = []
    for site in sites:
        # full local training
        # using all site info and correct modality training function
        updated_state, sample_count, ckpt_eva_results = local_training(global_state, site, freeze_global) 
        ckpt_eva_results["site_name"] = site["name"]
        
        # record the reaults
        global_updates_with_sample_count.append((updated_state, sample_count))
        # current_best_client_heads.append(best_head_state)
        # newest head, current best head recorded in .pth

        all_ckpt_eva_results.append(ckpt_eva_results)
    ################## END in sites, back to server ##################

    # 4. reutrns all trained global encoders from all sites
    if freeze_global:
        print("Finish head re-training")
    else:
        print("Finish site traininig")
    
    return global_updates_with_sample_count, all_ckpt_eva_results # ((state_dict, int:sample_count), list of dict)

In [12]:
def federated_ckpt_evaluation (current_results, best_results):
    # traverse through all sites
    new_best_found = False

    # 1. calculate the results
    # macro avg acc only (best for check point)
    current_results_df = pd.DataFrame(current_results)
    macro_acc_current  = current_results_df["val_acc"].mean()
    # weighted_acc = (current_results_df["val_acc"] * current_results_df["num_samples"]).sum() / current_results_df["num_samples"].sum()

    # 2. worst-site guard: Keep it only if no site acc dropped by more than 2 %.
    worst_acc_current = current_results_df["val_acc"].min()

    best_results_df = pd.DataFrame(best_results)
    macro_acc_best  = best_results_df["val_acc"].mean()
    worst_acc_best  = best_results_df["val_acc"].min()

    if macro_acc_current > macro_acc_best and worst_acc_current >= worst_acc_best - 0.02:
        # better macro avg & no sever single drop -> keep the result
        # record best scores
        best_results = current_results # list, not _df
        # tell the caller to save best global model (encoders)
        new_best_found = True
    else:
        # new model is not better
        new_best_found = False

    # 6. return the results
    return new_best_found, best_results

In [17]:
# 3. all sites preparations:
# FOR EACH SITE:
# 3.1 dataset preprocessing (remians unchanged over training)
# return: "clean_dataset_path": str, "label_col": str, "n_classes": int 
# 3.2 trai-val-test split (remians unchanged over training)
# 3.3 initialize local head (only ONCE)
# newest site head (used during training) + best site head (as outcome)
# 3.4 assign the correct local training function
# 3.5store all above info

def site_preparations (site):
    '''
    # 3.1 dataset preprocessing (remians unchanged over training)
    # return: "clean_dataset_path": str, "label_col": str, "n_classes": int
    # BEST SOLUTION: Find a way to record the preprocessing pipeline in site_info.json
    pass
        
    # placeholders:
    # site tabular_1
    site["clean_dataset"] = "tabular_dataset/diabetes_012_ready_to_model.csv"
    site["label_col"] = "Diabetes_012"
    site["n_classes"] = 2

    # 3.2 trai-val-test split (remians unchanged over training)
    pass
    # placeholders:
    site["clean_dataset_train"] = "tabular_dataset/diabetes_012_train.csv"
    site["clean_dataset_val"]   = "tabular_dataset/diabetes_012_val.csv"
    site["clean_dataset_test"]  = "tabular_dataset/diabetes_012_test.csv"
    '''
    
    # 3.3 initialize local head (only ONCE)
    # file / folder existing: impossible
    # NOT existing: initialize 
    model_name  = site["name"]
    modality    = site["modality"]
    n_classes   = site["n_classes"]
    
    # newest site head (used during training)
    folder_name = "newest_local_heads"
    site["newest_head"]       = build_local_head(folder_name, model_name, modality, n_classes)        
    
    # overall best site head (as outcome)
    folder_name = "overall_best_local_heads"
    site["overall_best_head"] = build_local_head(folder_name, model_name, modality, n_classes)   
    
    # current best site head (as outcome)
    folder_name = "current_best_local_heads"
    site["current_best_head"] = build_local_head(folder_name, model_name, modality, n_classes)
                
    # 3.4 assign the correct local training function
    site["site_training"] = assign_local_training_function(site["modality"])

    return site

In [14]:
# 0. (BEFORE training) clean the recorded local heads (newest)
folder_path = "newest_local_heads"
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)   # deletes the folder and everything inside
    print(f"Deleted folder: {folder_path}")
else:
    print("Folder does not exist.")

# clean the recorded local heads (best)
folder_path = "overall_best_local_heads"
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)   # deletes the folder and everything inside
    print(f"Deleted folder: {folder_path}")
else:
    print("Folder does not exist.")

folder_path = "current_best_local_heads"
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)   # deletes the folder and everything inside
    print(f"Deleted folder: {folder_path}")
else:
    print("Folder does not exist.")

Deleted folder: newest_local_heads
Deleted folder: overall_best_local_heads
Deleted folder: current_best_local_heads


In [15]:
# 1. initialize global model
global_encoders = SharedEncoders(
    d_tabular = D_TABULAR, 
    d_embedding = D_EMBEDDING, 
    d_fusion = D_FUSION
    )
global_state = global_encoders.state_dict()

In [18]:
# 2. build sites
# 2.1 run build site info (list of dict)
pass

# 2.2 import all sites (basic info)
# with open("sites_info_tabular.json", "r") as f:
with open("sites_info.json", "r") as f:
    sites = json.load(f)

sites

[{'name': 'tabular_1',
  'modality': 'tabular',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'tabular_dataset/tabular_1/diabetes_012_train.csv',
  'clean_dataset_val': 'tabular_dataset/tabular_1/diabetes_012_val.csv',
  'clean_dataset_test': 'tabular_dataset/tabular_1/diabetes_012_test.csv'},
 {'name': 'image_1',
  'modality': 'image',
  'raw_dataset_path': 'image_dataset/split',
  'n_classes': 5,
  'clean_dataset_train': 'imager_dataset/image_1/train',
  'clean_dataset_val': 'imager_dataset/image_1/val',
  'clean_dataset_test': 'imager_dataset/image_1/test'}]

In [19]:
# 3. all sites preparations:
# FOR EACH SITE:
# 3.1 dataset preprocessing (remians unchanged over training)
# return: "clean_dataset_path": str, "label_col": str, "n_classes": int
 
# 3.2 trai-val-test split (remians unchanged over training)

# 3.3 initialize local head (only ONCE)
# newest site head (used during training) + best site head (as outcome)

# 3.4 assign the correct local training function

# 3.5store all above info

for site in sites:
    site = site_preparations(site)

sites

[{'name': 'tabular_1',
  'modality': 'tabular',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'tabular_dataset/tabular_1/diabetes_012_train.csv',
  'clean_dataset_val': 'tabular_dataset/tabular_1/diabetes_012_val.csv',
  'clean_dataset_test': 'tabular_dataset/tabular_1/diabetes_012_test.csv',
  'newest_head': 'newest_local_heads\\tabular_1.pth',
  'overall_best_head': 'overall_best_local_heads\\tabular_1.pth',
  'current_best_head': 'current_best_local_heads\\tabular_1.pth',
  'site_training': <function tabular_site_training.training(global_state, freeze_global, train_set_path, val_set_path, labelcol, n_classes, newest_head_path, current_best_head_path)>},
 {'name': 'image_1',
  'modality': 'image',
  'raw_dataset_path': 'image_dataset/split',
  'n_classes': 5,
  'clean_dataset_train': 'imager_dataset/image_1/train',
  'clean_dataset_val': 'imager_dataset/image_1/val',
  'clean_dataset_test': 'imager_dataset/image_1/test',
  'newest_head': 'newest_local_head

In [290]:
# 4. operate federated training loop
# 4.0 configs
training_round = 5

# 4.1 initialization -- best global: save the initial global state as current best
best_global_path = "best_global_encoders.pth"
torch.save(global_state, best_global_path)

# 4.2 initialization -- best evaluation scores
best_ckpt_eva_results = [] # list of dict: {"site_name", "val_loss", "val_acc", "num_samples"}
for site in sites:
    results = {
        "site_name": site["name"],
        "val_loss" : float("inf"),
        "val_acc"  : -1.0,
        "num_samples": 0
        }
    best_ckpt_eva_results.append(results)

In [ ]:
# 4.3 training
for i in range(training_round):
    print("global update round {:d}:".format(i))
    # 1. one federated training round -> ALL sites FULLY trained ONCE
    global_updates_with_sample_count, _ = federated_training_one_round(global_state, sites, freeze_global=False)
    # ((state_dict, int:sample_count), list of dict)
    # current global state stays in memory, never saved in any files
    # ONLY current BEST global states be saved

    # 2. Server aggregates encoder weights using FedAvg (per key, selectively)
    agg_state = global_agg_fedavg(global_updates_with_sample_count)

    # 3. (OPTIONAL) head retraining with agg_state
    # freezed global encoders
    # double running time (do 1 samalier training round 2 times: 1 with freezed global encoders, and 1 without)
    # perfect match of global encoders and local heads
    _, current_ckpt_eva_results = federated_training_one_round(agg_state, sites, freeze_global=True)    
    # ((state_dict, int:sample_count), list of dict)

    # 4. Evaluation on val set
    new_best_found, best_site_evaluation_results = federated_ckpt_evaluation (current_ckpt_eva_results, 
                                                                              best_ckpt_eva_results
                                                                              )
    # best_site_evaluation_results updated by return values

    # 5. record the new model (global + all heads if they are the best)
    # (pravicy issues: server and site should save model components seperately)
    if new_best_found:
        # save the new_global_state to best_global_path
        torch.save(agg_state, best_global_path)
        # call all site to replace the "overall_best_local_heads" with "current_best_local_heads"
        pass
    else: # no changes
        pass
    
    # 4. start the next round
    global_state = agg_state

 [best updated] acc: 0.8360
 [best updated] acc: 0.8370
 [best updated] acc: 0.8373
tabular site training DONE
Finish site traininig
 [best updated] acc: 0.8375
tabular site training DONE
Finish head re-training
 [best updated] acc: 0.8363
 [best updated] acc: 0.8366
 [best updated] acc: 0.8373
tabular site training DONE
Finish site traininig
 [best updated] acc: 0.8368
tabular site training DONE
Finish head re-training
 [best updated] acc: 0.8363
 [best updated] acc: 0.8364
tabular site training DONE
Finish site traininig
 [best updated] acc: 0.8357
 [best updated] acc: 0.8359
 [best updated] acc: 0.8359
tabular site training DONE
Finish head re-training
 [best updated] acc: 0.8347
 [best updated] acc: 0.8349
tabular site training DONE
Finish site traininig
 [best updated] acc: 0.8346
 [best updated] acc: 0.8350
tabular site training DONE
Finish head re-training
 [best updated] acc: 0.8338
tabular site training DONE
Finish site traininig
 [best updated] acc: 0.8337
 [best updated] acc

In [ ]:
# 4. final evaluation